<!--    
    ================================================================================================
    
    Copyright (c) 2026 Ilya Snegov

    Licensed under the Apache License, Version 2.0 (the "License");
    you may not use this file except in compliance with the License.
    You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

    ================================================================================================
    notebooks/01-exploratory-data-analysis.ipynb
    ================================================================================================
-->

# **Exploratory Data Analysis**

*A Jupyter notebook for exploratory data analysis, target variable engineering, and data quality inspection from raw cryptocurrency OHLCV data.*

## **I. Setup the environment**

### **Import required libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from finml_lib import download_dataset_if_not_exists, set_matplotlib_style
from finml_lib.target import add_next_period_return_target, compute_acf_statistics

### **Apply global Matplotlib style for consistent visualization**

In [ ]:
set_matplotlib_style()

### **Get the raw cryptocurrency data**

In [ ]:
path = download_dataset_if_not_exists(
    handle="arthurneuron/cryptocurrency-futures-ohlcv-dataset-1m-2024", 
    filename="BTCUSDT.csv",
    output_dir="./data/"
)

### **Load the raw data**

In [ ]:
data_df = pd.read_csv(path)
data_df

## **II. Transform and aggregate raw data for task requirements**

### **Rename columns for clarity and transform datetime column into proper index format**

In [ ]:
data_df = data_df.rename(
    columns={
        'o': 'open', 
        'h': 'high', 
        'l': 'low', 
        'c': 'close', 
        'v': 'volume', 
        'd': 'datetime'
    }
)

data_df['datetime'] = pd.to_datetime(data_df['datetime'], unit='ms')
data_df = data_df.set_index('datetime')

data_df

### **Aggregate 1-minute data into 1-hour candles**

In [ ]:
aggregated_data_df = data_df.resample('1h').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum',
})

aggregated_data_df

## **III. Engineer the target variable as the next-period return**

In [ ]:
data_with_target_df = add_next_period_return_target(aggregated_data_df)
data_with_target_df

## **IV. Inspect the target variable**

### **Extract and prepare the target variable for statistical inspection**

In [ ]:
target_series = data_with_target_df['target']
target_np_array = target_series.to_numpy()

### **Inspect the temporal dynamics of the target variable**

In [ ]:
plt.plot(target_np_array, color='blue')

plt.title(f'Target Variable Over Time')
plt.xlabel('Time Steps')
plt.ylabel('Next-Period Return')
plt.show()

### **Inspect the distribution of the target variable**

In [ ]:
sns.histplot(
    target_np_array,
    bins=100,
    color="green",
    edgecolor="black",
    stat="density",
    alpha=0.5,
    kde=True
)

plt.axvline(x=0, color='red', linestyle='--')

plt.title("Distribution of Target Variable (Next-Period Returns)")
plt.xlabel("Return Value")
plt.ylabel("Density")
plt.show()

### **Inspect the descriptive statistics of the distribution of the target variable**

In [ ]:
stats_df = pd.DataFrame({
    'Metric': pd.Series(
        [
            'Total samples',
            'NaN count',
            'Mean',
            'Std Dev',
            'Min',
            'Max',
            'Skewness',
            'Kurtosis (excess)',
            'Zero-return ratio (%)'
        ], 
        dtype=object
    ),
    'Value': pd.Series(
        [
            len(target_series),
            target_series.isna().sum(),
            target_series.mean(),
            target_series.std(),
            target_series.min(),
            target_series.max(),
            target_series.skew(),
            target_series.kurtosis(),
            (target_series == 0).mean() * 100
        ], 
        dtype=object
    )
})

stats_df

### **Inspect the quantile distribution of the target variable**

In [ ]:
quantiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

quantiles_df = pd.DataFrame({
    'Quantile': pd.Series(
        [f'Q{q:.2f}' for q in quantiles], 
        dtype=object
    ),
    'Value': pd.Series(
        [target_series.quantile(q) for q in quantiles],
        dtype='Float64'
    ),
})

quantiles_df

### **Inspect the extreme values of the target variable**

In [ ]:
lower_quantile = 0.01
upper_quantile = 0.99

q_low = target_series.quantile(lower_quantile)
q_high = target_series.quantile(upper_quantile)

is_extreme = (
    (target_series < q_low)
    | (target_series > q_high)
)

colors = np.where(is_extreme, "red", "blue")

fig, ax = plt.subplots()

ax.scatter(
    target_series.index,
    target_series,
    c=colors,
    s=1,
    alpha=1,
)

ax.axhline(q_low, color="red", linestyle="--", alpha=0.7)
ax.axhline(q_high, color="red", linestyle="--", alpha=0.7)

ax.set_xlabel("Time")
ax.set_ylabel("Target Value (Return)")
ax.set_title("Target Variable Over Time (Extremes in Red)")

plt.show()

### **Inspect the autocorrelation of the target variable at various time lags**

In [ ]:
compute_acf_statistics(target_series, 24)

In [ ]:
compute_acf_statistics(target_series, 72)

In [ ]:
compute_acf_statistics(target_series, 168)

## **V. Analysis of target variable distribution and temporal dynamics**

### **Overview**

The target variable exhibits extreme leptokurtosis (excess kurtosis $\kappa = 50.56$) and pronounced heteroskedasticity. Extreme returns are heavily concentrated in the initial period (2019–2020) and decay over time, indicating non-stationary variance. The autocorrelation function (ACF) shows negligible linear dependence ($\max |ACF| \approx 0.03$), confirming that the target behaves as a heavy-tailed white noise process. In regression tasks, such extreme outliers disproportionately dominate the loss landscape, forcing the model to prioritize rare, high-magnitude shocks over typical market patterns.

### **Evaluation of handling strategies**

| Strategy | Mechanism |
| :--- | :--- |
| **Row Deletion** | Removes observations exceeding predefined thresholds. |
| **StandardScaler** | Centers and scales data using the mean and standard deviation. |
| **MinMaxScaler** | Scales data to a fixed range using absolute minimum and maximum values. |
| **RobustScaler** | Centers and scales data using the median and interquartile range (IQR). |
| **Clipping (Winsorization)** | Caps values at predefined quantile thresholds (e.g., 1st and 99th percentiles). |

### **Conclusion**

In regression tasks, handling extreme outliers requires careful consideration of the method's impact on the data distribution. **Row Deletion** is fundamentally unsuitable as it disrupts the temporal continuity and sequential structure of the time series, introducing gaps that invalidate time-dependent modeling. Similarly, standard scaling methods fail to address the heavy tails: **StandardScaler** relies on the mean and standard deviation, which are heavily skewed by outliers, compressing the central 99% of the data into a negligible numerical range. **MinMaxScaler** is equally flawed, as it relies on absolute extremes; a single massive outlier compresses the entire normal distribution, rendering standard variations numerically indistinguishable.

While **RobustScaler** is partially suitable because it preserves the scale of the central distribution using the median and interquartile range (IQR), it leaves extreme outliers unbounded. This fails to mitigate the sensitivity of standard loss functions to massive errors. Furthermore, although robust loss functions like the Huber loss ($\mathcal{L}_{Huber}$) reduce the gradient penalty for large residuals compared to the standard Mean Squared Error ($\mathcal{L}_{MSE}$), they do not fully neutralize the non-stationary volatility observed in the initial period.

Therefore, **Clipping (Winsorization)** at the 1st and 99th percentiles is the optimal and strictly applied prerequisite. It directly neutralizes the disproportionate influence of extreme shocks while preserving the variance structure of the typical market regime. This explicitly bounds the target space, ensuring the model learns the central distribution's dynamics without being derailed by unbounded historical anomalies.

## **VI. Winsorize the target variable at the 1st and 99th percentiles**

In [ ]:
lower_threshold = data_with_target_df['target'].quantile(0.01)
upper_threshold = data_with_target_df['target'].quantile(0.99)

data_with_target_df['target'] = data_with_target_df['target'].clip(
    lower=lower_threshold,
    upper=upper_threshold
)

data_with_target_df

## **VII. Prepare the dataset for feature engineering and modeling**

### **Chronologically split the data into 70% train, 15% validation, and 15% test sets**

In [ ]:
n = len(data_with_target_df)

train_end_idx = int(n * 0.70)
valid_end_idx = int(n * 0.85)

train_df = data_with_target_df.iloc[:train_end_idx]
valid_df = data_with_target_df.iloc[train_end_idx:valid_end_idx]
test_df = data_with_target_df.iloc[valid_end_idx:]

### **Serialize the partitioned datasets to Parquet format for efficient storage**

In [ ]:
train_df.to_parquet("./data/train.parquet")
valid_df.to_parquet("./data/valid.parquet")
test_df.to_parquet("./data/test.parquet")